In [1]:
x = '4D'  # No of days of interval
y = 30 #no of predictions

In [2]:
open_preds = []
high_preds = []
low_preds = []
close_preds = []
volume_preds = []
turnover_preds = [] 

In [3]:
# ============================================================
# 1️⃣ IMPORT LIBRARIES
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import seaborn as sns
warnings.filterwarnings("ignore")

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, precision_score, recall_score, f1_score, accuracy_score, confusion_matrix

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, SimpleRNN
from tensorflow.keras.callbacks import EarlyStopping

from tabulate import tabulate

## for close price

In [4]:
# ============================================================
# 2️⃣ LOAD & PREPROCESS MULTI-YEAR NSE DATASET
# ============================================================

df = pd.read_csv("yahoo_data.csv", encoding="utf-8-sig")

# Clean columns
df.columns = df.columns.str.strip()

# Convert date
df['Date'] = pd.to_datetime(df['Date'])

# Sort ascending
df = df.sort_values("Date")

# Set index
df.set_index("Date", inplace=True)

# Use Close price
data = df[['Close']]

# Train-Test Split (80-20)
train_size = int(len(data) * 0.8)
train, test = data[:train_size], data[train_size:]

print("Training size:", len(train))
print("Testing size :", len(test))

Training size: 1006
Testing size : 252


In [5]:
# ============================================================
# 4️⃣ HELPER FUNCTIONS
# ============================================================

def create_sequences(data, time_steps=60):
    X, y = [], []
    for i in range(len(data) - time_steps):
        X.append(data[i:i+time_steps])
        y.append(data[i+time_steps])
    return np.array(X), np.array(y)


def evaluate_model(actual, predicted, name):
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae = mean_absolute_error(actual, predicted)
    r2 = r2_score(actual, predicted)

    print(f"\n{name}")
    print("RMSE:", rmse)
    print("MAE :", mae)
    print("R2  :", r2)

    return rmse, mae, r2


time_steps = 60

In [6]:
import numpy as np

def predict_future_prices(model, last_data, scaler, time_steps, future_days):
    
    # Ensure correct shape
    input_seq = last_data[-time_steps:]
    input_seq = input_seq.reshape(1, time_steps, 1)

    future_predictions = []

    for _ in range(future_days):
        
        pred = model.predict(input_seq, verbose=0)
        
        # Store prediction
        future_predictions.append(pred[0, 0])
        
        # Reshape prediction properly to (1,1,1)
        pred_reshaped = pred.reshape(1, 1, 1)
        
        # Remove first timestep and append prediction
        input_seq = np.concatenate(
            (input_seq[:, 1:, :], pred_reshaped),
            axis=1
        )

    # Convert back to original scale
    future_predictions = scaler.inverse_transform(
        np.array(future_predictions).reshape(-1, 1)
    )

    return future_predictions.flatten()

In [7]:
scaler = MinMaxScaler()

train_scaled = scaler.fit_transform(train)
test_scaled = scaler.transform(test)

In [8]:
time_steps = 30

# --- SARIMA ---
sarima_model = SARIMAX(train['Close'],
                       order=(4,0,3),
                       seasonal_order=(1,0,1,5))

sarima_result = sarima_model.fit()

forecast = sarima_result.forecast(steps=len(test))

# --- Residuals ---
residuals = test['Close'].values - forecast.values
residuals = residuals.reshape(-1,1)

# --- Scaling ---
from sklearn.preprocessing import StandardScaler
scaler_res = StandardScaler()
res_scaled = scaler_res.fit_transform(residuals)

X_res, y_res = create_sequences(res_scaled, time_steps)

# --- RNN ---
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

model = Sequential([
    SimpleRNN(61, return_sequences=True, input_shape=(time_steps,1)),
    Dropout(0.2),
    SimpleRNN(32),
    Dense(1)
])

model.compile(optimizer=Adam(0.0005), loss='huber')

early_stop = EarlyStopping(patience=10, restore_best_weights=True)

model.fit(X_res, y_res,
          epochs=150,
          batch_size=16,
          callbacks=[early_stop],
          verbose=0)

# --- Prediction ---
pred = model.predict(X_res)
pred = scaler_res.inverse_transform(pred)

final_pred = forecast[time_steps:].values + pred.flatten()
actual = test['Close'].values[time_steps:]

C:\Users\Asus\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\Asus\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


C:\Users\Asus\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\Asus\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(



7/7 [==============================] - 0s 3ms/step


In [9]:
future_days = y
future_prices = predict_future_prices(
    model,
    test_scaled,
    scaler,
    time_steps,
    future_days
)

print("Future Prices:", future_prices)

Future Prices: [32949.617 32369.953 31434.066 31030.074 30679.096 30389.094 30484.924
 30499.643 30530.488 30429.422 30159.322 29822.713 29435.986 29207.896
 29216.803 29533.3   29719.182 30137.82  30323.857 30706.791 30755.576
 30289.395 29997.967 29727.998 30010.127 30291.838 31118.422 31531.096
 32058.55  31966.43 ]


In [10]:
last_date = df.index[-1]
future_dates = pd.date_range(last_date, periods=future_days+1, freq=x)[1:]

future_df = pd.DataFrame({
    "Date": future_dates,
    "Predicted_Price": future_prices
})

print(future_df)

         Date  Predicted_Price
0  2023-05-02     32949.617188
1  2023-05-06     32369.953125
2  2023-05-10     31434.066406
3  2023-05-14     31030.074219
4  2023-05-18     30679.095703
5  2023-05-22     30389.093750
6  2023-05-26     30484.923828
7  2023-05-30     30499.642578
8  2023-06-03     30530.488281
9  2023-06-07     30429.421875
10 2023-06-11     30159.322266
11 2023-06-15     29822.712891
12 2023-06-19     29435.986328
13 2023-06-23     29207.896484
14 2023-06-27     29216.802734
15 2023-07-01     29533.300781
16 2023-07-05     29719.181641
17 2023-07-09     30137.820312
18 2023-07-13     30323.857422
19 2023-07-17     30706.791016
20 2023-07-21     30755.576172
21 2023-07-25     30289.394531
22 2023-07-29     29997.966797
23 2023-08-02     29727.998047
24 2023-08-06     30010.126953
25 2023-08-10     30291.837891
26 2023-08-14     31118.421875
27 2023-08-18     31531.095703
28 2023-08-22     32058.550781
29 2023-08-26     31966.429688


In [11]:
close_preds = future_prices.copy()

## For Open Price

In [12]:
# ============================================================
# 2️⃣ LOAD & PREPROCESS MULTI-YEAR NSE DATASET
# ============================================================

df = pd.read_csv("yahoo_data.csv", encoding="utf-8-sig")

# Clean columns
df.columns = df.columns.str.strip()

# Convert date
df['Date'] = pd.to_datetime(df['Date'])

# Sort ascending
df = df.sort_values("Date")

# Set index
df.set_index("Date", inplace=True)

# Use Close price
data = df[['Open']]

# Train-Test Split (80-20)
train_size = int(len(data) * 0.8)
train, test = data[:train_size], data[train_size:]

print("Training size:", len(train))
print("Testing size :", len(test))

Training size: 1006
Testing size : 252


In [13]:
import numpy as np

def predict_future_prices(model, last_data, scaler, time_steps, future_days):
    
    # Ensure correct shape
    input_seq = last_data[-time_steps:]
    input_seq = input_seq.reshape(1, time_steps, 1)

    future_predictions = []

    for _ in range(future_days):
        
        pred = model.predict(input_seq, verbose=0)
        
        # Store prediction
        future_predictions.append(pred[0, 0])
        
        # Reshape prediction properly to (1,1,1)
        pred_reshaped = pred.reshape(1, 1, 1)
        
        # Remove first timestep and append prediction
        input_seq = np.concatenate(
            (input_seq[:, 1:, :], pred_reshaped),
            axis=1
        )

    # Convert back to original scale
    future_predictions = scaler.inverse_transform(
        np.array(future_predictions).reshape(-1, 1)
    )

    return future_predictions.flatten()

In [14]:
scaler = MinMaxScaler()

train_scaled = scaler.fit_transform(train)
test_scaled = scaler.transform(test)

In [15]:
time_steps = 30

# --- SARIMA ---
sarima_model = SARIMAX(train['Open'],
                       order=(4,0,3),
                       seasonal_order=(1,0,1,5))

sarima_result = sarima_model.fit()

forecast = sarima_result.forecast(steps=len(test))

# --- Residuals ---
residuals = test['Open'].values - forecast.values
residuals = residuals.reshape(-1,1)

# --- Scaling ---
from sklearn.preprocessing import StandardScaler
scaler_res = StandardScaler()
res_scaled = scaler_res.fit_transform(residuals)

X_res, y_res = create_sequences(res_scaled, time_steps)

# --- RNN ---
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

model = Sequential([
    SimpleRNN(61, return_sequences=True, input_shape=(time_steps,1)),
    Dropout(0.2),
    SimpleRNN(32),
    Dense(1)
])

model.compile(optimizer=Adam(0.0005), loss='huber')

early_stop = EarlyStopping(patience=10, restore_best_weights=True)

model.fit(X_res, y_res,
          epochs=150,
          batch_size=16,
          callbacks=[early_stop],
          verbose=0)

# --- Prediction ---
pred = model.predict(X_res)
pred = scaler_res.inverse_transform(pred)

final_pred = forecast[time_steps:].values + pred.flatten()
actual = test['Open'].values[time_steps:]

C:\Users\Asus\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\Asus\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\Asus\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\Asus\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will 

7/7 [==============================] - 0s 3ms/step


In [16]:
future_days = y
future_prices = predict_future_prices(
    model,
    test_scaled,
    scaler,
    time_steps,
    future_days
)

print("Future Prices:", future_prices)

Future Prices: [32793.082 32527.066 32198.07  31373.06  31093.373 30693.945 30081.371
 29805.414 29523.799 29045.748 28471.643 28208.537 27599.562 26608.268
 26263.887 25711.82  25204.17  24870.559 24997.998 24754.803 24606.197
 25072.59  24521.957 24128.61  24741.385 24065.277 24092.768 25011.166
 25311.926 25847.525]


In [17]:
last_date = df.index[-1]
future_dates = pd.date_range(last_date, periods=future_days+1, freq=x)[1:]

future_df = pd.DataFrame({
    "Date": future_dates,
    "Predicted_Price": future_prices
})

print(future_df)

         Date  Predicted_Price
0  2023-05-02     32793.082031
1  2023-05-06     32527.066406
2  2023-05-10     32198.070312
3  2023-05-14     31373.060547
4  2023-05-18     31093.373047
5  2023-05-22     30693.945312
6  2023-05-26     30081.371094
7  2023-05-30     29805.414062
8  2023-06-03     29523.798828
9  2023-06-07     29045.748047
10 2023-06-11     28471.642578
11 2023-06-15     28208.537109
12 2023-06-19     27599.562500
13 2023-06-23     26608.267578
14 2023-06-27     26263.886719
15 2023-07-01     25711.820312
16 2023-07-05     25204.169922
17 2023-07-09     24870.558594
18 2023-07-13     24997.998047
19 2023-07-17     24754.802734
20 2023-07-21     24606.197266
21 2023-07-25     25072.589844
22 2023-07-29     24521.957031
23 2023-08-02     24128.609375
24 2023-08-06     24741.384766
25 2023-08-10     24065.277344
26 2023-08-14     24092.767578
27 2023-08-18     25011.166016
28 2023-08-22     25311.925781
29 2023-08-26     25847.525391


In [18]:
open_preds = future_prices.copy()

## For High

In [19]:
# ============================================================
# 2️⃣ LOAD & PREPROCESS MULTI-YEAR NSE DATASET
# ============================================================

df = pd.read_csv("yahoo_data.csv", encoding="utf-8-sig")

# Clean columns
df.columns = df.columns.str.strip()

# Convert date
df['Date'] = pd.to_datetime(df['Date'])

# Sort ascending
df = df.sort_values("Date")

# Set index
df.set_index("Date", inplace=True)

# Use Close price
data = df[['High']]

# Train-Test Split (80-20)
train_size = int(len(data) * 0.8)
train, test = data[:train_size], data[train_size:]

print("Training size:", len(train))
print("Testing size :", len(test))

Training size: 1006
Testing size : 252


In [20]:
import numpy as np

def predict_future_prices(model, last_data, scaler, time_steps, future_days):
    
    # Ensure correct shape
    input_seq = last_data[-time_steps:]
    input_seq = input_seq.reshape(1, time_steps, 1)

    future_predictions = []

    for _ in range(future_days):
        
        pred = model.predict(input_seq, verbose=0)
        
        # Store prediction
        future_predictions.append(pred[0, 0])
        
        # Reshape prediction properly to (1,1,1)
        pred_reshaped = pred.reshape(1, 1, 1)
        
        # Remove first timestep and append prediction
        input_seq = np.concatenate(
            (input_seq[:, 1:, :], pred_reshaped),
            axis=1
        )

    # Convert back to original scale
    future_predictions = scaler.inverse_transform(
        np.array(future_predictions).reshape(-1, 1)
    )

    return future_predictions.flatten()

In [21]:
scaler = MinMaxScaler()

train_scaled = scaler.fit_transform(train)
test_scaled = scaler.transform(test)

In [22]:
time_steps = 30

# --- SARIMA ---
sarima_model = SARIMAX(train['High'],
                       order=(4,0,3),
                       seasonal_order=(1,0,1,5))

sarima_result = sarima_model.fit()

forecast = sarima_result.forecast(steps=len(test))

# --- Residuals ---
residuals = test['High'].values - forecast.values
residuals = residuals.reshape(-1,1)

# --- Scaling ---
from sklearn.preprocessing import StandardScaler
scaler_res = StandardScaler()
res_scaled = scaler_res.fit_transform(residuals)

X_res, y_res = create_sequences(res_scaled, time_steps)

# --- RNN ---
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

model = Sequential([
    SimpleRNN(61, return_sequences=True, input_shape=(time_steps,1)),
    Dropout(0.2),
    SimpleRNN(32),
    Dense(1)
])

model.compile(optimizer=Adam(0.0005), loss='huber')

early_stop = EarlyStopping(patience=10, restore_best_weights=True)

model.fit(X_res, y_res,
          epochs=150,
          batch_size=16,
          callbacks=[early_stop],
          verbose=0)

# --- Prediction ---
pred = model.predict(X_res)
pred = scaler_res.inverse_transform(pred)

final_pred = forecast[time_steps:].values + pred.flatten()
actual = test['High'].values[time_steps:]

C:\Users\Asus\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\Asus\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\Asus\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\Asus\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will 

7/7 [==============================] - 0s 3ms/step


In [23]:
future_days = y
future_prices = predict_future_prices(
    model,
    test_scaled,
    scaler,
    time_steps,
    future_days
)

print("Future Prices:", future_prices)

Future Prices: [35464.22  36652.074 37505.17  37743.973 37628.07  37287.008 37079.914
 37244.83  37698.227 38157.68  38363.9   38224.34  37903.21  37627.293
 37526.06  37611.555 37681.24  37647.516 37213.414 36679.113 36275.254
 36200.957 36381.996 36557.52  36514.86  36205.414 35781.49  35322.15
 35252.164 35531.56 ]


In [24]:
last_date = df.index[-1]
future_dates = pd.date_range(last_date, periods=future_days+1, freq=x)[1:]

future_df = pd.DataFrame({
    "Date": future_dates,
    "Predicted_Price": future_prices
})

print(future_df)

         Date  Predicted_Price
0  2023-05-02     35464.218750
1  2023-05-06     36652.074219
2  2023-05-10     37505.171875
3  2023-05-14     37743.972656
4  2023-05-18     37628.070312
5  2023-05-22     37287.007812
6  2023-05-26     37079.914062
7  2023-05-30     37244.828125
8  2023-06-03     37698.226562
9  2023-06-07     38157.679688
10 2023-06-11     38363.898438
11 2023-06-15     38224.339844
12 2023-06-19     37903.210938
13 2023-06-23     37627.292969
14 2023-06-27     37526.058594
15 2023-07-01     37611.554688
16 2023-07-05     37681.238281
17 2023-07-09     37647.515625
18 2023-07-13     37213.414062
19 2023-07-17     36679.113281
20 2023-07-21     36275.253906
21 2023-07-25     36200.957031
22 2023-07-29     36381.996094
23 2023-08-02     36557.519531
24 2023-08-06     36514.859375
25 2023-08-10     36205.414062
26 2023-08-14     35781.488281
27 2023-08-18     35322.148438
28 2023-08-22     35252.164062
29 2023-08-26     35531.558594


In [25]:
high_preds = future_prices.copy()

## For Low

In [26]:
# ============================================================
# 2️⃣ LOAD & PREPROCESS MULTI-YEAR NSE DATASET
# ============================================================

df = pd.read_csv("yahoo_data.csv", encoding="utf-8-sig")

# Clean columns
df.columns = df.columns.str.strip()

# Convert date
df['Date'] = pd.to_datetime(df['Date'])

# Sort ascending
df = df.sort_values("Date")

# Set index
df.set_index("Date", inplace=True)

# Use Close price
data = df[['Low']]

# Train-Test Split (80-20)
train_size = int(len(data) * 0.8)
train, test = data[:train_size], data[train_size:]

print("Training size:", len(train))
print("Testing size :", len(test))

Training size: 1006
Testing size : 252


In [27]:
import numpy as np

def predict_future_prices(model, last_data, scaler, time_steps, future_days):
    
    # Ensure correct shape
    input_seq = last_data[-time_steps:]
    input_seq = input_seq.reshape(1, time_steps, 1)

    future_predictions = []

    for _ in range(future_days):
        
        pred = model.predict(input_seq, verbose=0)
        
        # Store prediction
        future_predictions.append(pred[0, 0])
        
        # Reshape prediction properly to (1,1,1)
        pred_reshaped = pred.reshape(1, 1, 1)
        
        # Remove first timestep and append prediction
        input_seq = np.concatenate(
            (input_seq[:, 1:, :], pred_reshaped),
            axis=1
        )

    # Convert back to original scale
    future_predictions = scaler.inverse_transform(
        np.array(future_predictions).reshape(-1, 1)
    )

    return future_predictions.flatten()

In [28]:
scaler = MinMaxScaler()

train_scaled = scaler.fit_transform(train)
test_scaled = scaler.transform(test)

In [29]:
time_steps = 30

# --- SARIMA ---
sarima_model = SARIMAX(train['Low'],
                       order=(4,0,3),
                       seasonal_order=(1,0,1,5))

sarima_result = sarima_model.fit()

forecast = sarima_result.forecast(steps=len(test))

# --- Residuals ---
residuals = test['Low'].values - forecast.values
residuals = residuals.reshape(-1,1)

# --- Scaling ---
from sklearn.preprocessing import StandardScaler
scaler_res = StandardScaler()
res_scaled = scaler_res.fit_transform(residuals)

X_res, y_res = create_sequences(res_scaled, time_steps)

# --- RNN ---
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

model = Sequential([
    SimpleRNN(61, return_sequences=True, input_shape=(time_steps,1)),
    Dropout(0.2),
    SimpleRNN(32),
    Dense(1)
])

model.compile(optimizer=Adam(0.0005), loss='huber')

early_stop = EarlyStopping(patience=10, restore_best_weights=True)

model.fit(X_res, y_res,
          epochs=150,
          batch_size=16,
          callbacks=[early_stop],
          verbose=0)

# --- Prediction ---
pred = model.predict(X_res)
pred = scaler_res.inverse_transform(pred)

final_pred = forecast[time_steps:].values + pred.flatten()
actual = test['Low'].values[time_steps:]

C:\Users\Asus\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\Asus\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\Asus\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\Asus\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will 

7/7 [==============================] - 0s 3ms/step


In [30]:
future_days = y
future_prices = predict_future_prices(
    model,
    test_scaled,
    scaler,
    time_steps,
    future_days
)

print("Future Prices:", future_prices)

Future Prices: [34162.65  34702.766 34966.133 35022.344 34859.242 34614.965 34416.41
 34380.137 34422.688 34469.32  34519.043 34506.71  34496.06  34523.098
 34575.508 34623.49  34525.324 34290.555 33990.55  33806.305 33800.95
 33978.04  34185.17  34367.27  34407.56  34264.25  34025.254 33844.645
 33781.246 33810.77 ]


In [31]:
last_date = df.index[-1]
future_dates = pd.date_range(last_date, periods=future_days+1, freq=x)[1:]

future_df = pd.DataFrame({
    "Date": future_dates,
    "Predicted_Price": future_prices
})

print(future_df)

         Date  Predicted_Price
0  2023-05-02     34162.648438
1  2023-05-06     34702.765625
2  2023-05-10     34966.132812
3  2023-05-14     35022.343750
4  2023-05-18     34859.242188
5  2023-05-22     34614.964844
6  2023-05-26     34416.410156
7  2023-05-30     34380.136719
8  2023-06-03     34422.687500
9  2023-06-07     34469.320312
10 2023-06-11     34519.042969
11 2023-06-15     34506.710938
12 2023-06-19     34496.058594
13 2023-06-23     34523.097656
14 2023-06-27     34575.507812
15 2023-07-01     34623.488281
16 2023-07-05     34525.324219
17 2023-07-09     34290.554688
18 2023-07-13     33990.550781
19 2023-07-17     33806.304688
20 2023-07-21     33800.949219
21 2023-07-25     33978.039062
22 2023-07-29     34185.171875
23 2023-08-02     34367.269531
24 2023-08-06     34407.558594
25 2023-08-10     34264.250000
26 2023-08-14     34025.253906
27 2023-08-18     33844.644531
28 2023-08-22     33781.246094
29 2023-08-26     33810.769531


In [32]:
low_preds = future_prices.copy()

In [33]:
import numpy as np

def predict_future_prices(model, last_data, scaler, time_steps, future_days):
    
    # Ensure correct shape
    input_seq = last_data[-time_steps:]
    input_seq = input_seq.reshape(1, time_steps, 1)

    future_predictions = []

    for _ in range(future_days):
        
        pred = model.predict(input_seq, verbose=0)
        
        # Store prediction
        future_predictions.append(pred[0, 0])
        
        # Reshape prediction properly to (1,1,1)
        pred_reshaped = pred.reshape(1, 1, 1)
        
        # Remove first timestep and append prediction
        input_seq = np.concatenate(
            (input_seq[:, 1:, :], pred_reshaped),
            axis=1
        )

    # Convert back to original scale
    future_predictions = scaler.inverse_transform(
        np.array(future_predictions).reshape(-1, 1)
    )

    return future_predictions.flatten()

In [36]:
future_days = y
future_prices = predict_future_prices(
    model,
    test_scaled,
    scaler,
    time_steps,
    future_days
)

print("Future Prices:", future_prices)

Future Prices: [34162.65  34702.766 34966.133 35022.344 34859.242 34614.965 34416.41
 34380.137 34422.688 34469.32  34519.043 34506.71  34496.06  34523.098
 34575.508 34623.49  34525.324 34290.555 33990.55  33806.305 33800.95
 33978.04  34185.17  34367.27  34407.56  34264.25  34025.254 33844.645
 33781.246 33810.77 ]


In [37]:
last_date = df.index[-1]
future_dates = pd.date_range(last_date, periods=future_days+1, freq=x)[1:]

future_df = pd.DataFrame({
    "Date": future_dates,
    "Predicted_Price": future_prices
})

print(future_df)

         Date  Predicted_Price
0  2023-05-02     34162.648438
1  2023-05-06     34702.765625
2  2023-05-10     34966.132812
3  2023-05-14     35022.343750
4  2023-05-18     34859.242188
5  2023-05-22     34614.964844
6  2023-05-26     34416.410156
7  2023-05-30     34380.136719
8  2023-06-03     34422.687500
9  2023-06-07     34469.320312
10 2023-06-11     34519.042969
11 2023-06-15     34506.710938
12 2023-06-19     34496.058594
13 2023-06-23     34523.097656
14 2023-06-27     34575.507812
15 2023-07-01     34623.488281
16 2023-07-05     34525.324219
17 2023-07-09     34290.554688
18 2023-07-13     33990.550781
19 2023-07-17     33806.304688
20 2023-07-21     33800.949219
21 2023-07-25     33978.039062
22 2023-07-29     34185.171875
23 2023-08-02     34367.269531
24 2023-08-06     34407.558594
25 2023-08-10     34264.250000
26 2023-08-14     34025.253906
27 2023-08-18     33844.644531
28 2023-08-22     33781.246094
29 2023-08-26     33810.769531


In [38]:
volume_preds = future_prices.copy()

In [40]:
import numpy as np

def predict_future_prices(model, last_data, scaler, time_steps, future_days):
    
    # Ensure correct shape
    input_seq = last_data[-time_steps:]
    input_seq = input_seq.reshape(1, time_steps, 1)

    future_predictions = []

    for _ in range(future_days):
        
        pred = model.predict(input_seq, verbose=0)
        
        # Store prediction
        future_predictions.append(pred[0, 0])
        
        # Reshape prediction properly to (1,1,1)
        pred_reshaped = pred.reshape(1, 1, 1)
        
        # Remove first timestep and append prediction
        input_seq = np.concatenate(
            (input_seq[:, 1:, :], pred_reshaped),
            axis=1
        )

    # Convert back to original scale
    future_predictions = scaler.inverse_transform(
        np.array(future_predictions).reshape(-1, 1)
    )

    return future_predictions.flatten()

In [41]:
scaler = MinMaxScaler()

train_scaled = scaler.fit_transform(train)
test_scaled = scaler.transform(test)

In [43]:
future_days = y
future_prices = predict_future_prices(
    model,
    test_scaled,
    scaler,
    time_steps,
    future_days
)

print("Future Prices:", future_prices)

Future Prices: [34162.65  34702.766 34966.133 35022.344 34859.242 34614.965 34416.41
 34380.137 34422.688 34469.32  34519.043 34506.71  34496.06  34523.098
 34575.508 34623.49  34525.324 34290.555 33990.55  33806.305 33800.95
 33978.04  34185.17  34367.27  34407.56  34264.25  34025.254 33844.645
 33781.246 33810.77 ]


In [44]:
last_date = df.index[-1]
future_dates = pd.date_range(last_date, periods=future_days+1, freq=x)[1:]

future_df = pd.DataFrame({
    "Date": future_dates,
    "Predicted_Price": future_prices
})

print(future_df)

         Date  Predicted_Price
0  2023-05-02     34162.648438
1  2023-05-06     34702.765625
2  2023-05-10     34966.132812
3  2023-05-14     35022.343750
4  2023-05-18     34859.242188
5  2023-05-22     34614.964844
6  2023-05-26     34416.410156
7  2023-05-30     34380.136719
8  2023-06-03     34422.687500
9  2023-06-07     34469.320312
10 2023-06-11     34519.042969
11 2023-06-15     34506.710938
12 2023-06-19     34496.058594
13 2023-06-23     34523.097656
14 2023-06-27     34575.507812
15 2023-07-01     34623.488281
16 2023-07-05     34525.324219
17 2023-07-09     34290.554688
18 2023-07-13     33990.550781
19 2023-07-17     33806.304688
20 2023-07-21     33800.949219
21 2023-07-25     33978.039062
22 2023-07-29     34185.171875
23 2023-08-02     34367.269531
24 2023-08-06     34407.558594
25 2023-08-10     34264.250000
26 2023-08-14     34025.253906
27 2023-08-18     33844.644531
28 2023-08-22     33781.246094
29 2023-08-26     33810.769531


In [45]:
turnover_preds = future_prices.copy()

In [46]:
from tabulate import tabulate

# Create DataFrame
future_df = pd.DataFrame({
    "Date": future_dates,
    "Open": open_preds,
    "High": high_preds,
    "Low": low_preds,
    "Close": close_preds,
    "Shares Traded": volume_preds,
    #"Turnover": turnover_preds
})

print(tabulate(future_df, headers='keys', tablefmt='pretty', showindex=False))

+---------------------+-----------------+----------------+----------------+-----------------+----------------+
|        Date         |      Open       |      High      |      Low       |      Close      | Shares Traded  |
+---------------------+-----------------+----------------+----------------+-----------------+----------------+
| 2023-05-02 00:00:00 | 32793.08203125  |  35464.21875   | 34162.6484375  |  32949.6171875  | 34162.6484375  |
| 2023-05-06 00:00:00 | 32527.06640625  | 36652.07421875 |  34702.765625  |  32369.953125   |  34702.765625  |
| 2023-05-10 00:00:00 |  32198.0703125  |  37505.171875  | 34966.1328125  | 31434.06640625  | 34966.1328125  |
| 2023-05-14 00:00:00 | 31373.060546875 | 37743.97265625 |  35022.34375   | 31030.07421875  |  35022.34375   |
| 2023-05-18 00:00:00 | 31093.373046875 | 37628.0703125  | 34859.2421875  | 30679.095703125 | 34859.2421875  |
| 2023-05-22 00:00:00 |  30693.9453125  | 37287.0078125  | 34614.96484375 |   30389.09375   | 34614.96484375 |
|